In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df=pd.read_json('../original_datasets/product_catalog.json')

print(df.info())
print(df.describe())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 878 entries, 0 to 877
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   catalog_version  878 non-null    str                
 1   generated_at     878 non-null    datetime64[us, UTC]
 2   record_count     878 non-null    int64              
 3   products         878 non-null    object             
dtypes: datetime64[us, UTC](1), int64(1), object(1), str(1)
memory usage: 31.1+ KB
None
       record_count
count         878.0
mean          878.0
std             0.0
min           878.0
25%           878.0
50%           878.0
75%           878.0
max           878.0
  catalog_version              generated_at  record_count  \
0            v3.2 2025-07-01 00:00:00+00:00           878   
1            v3.2 2025-07-01 00:00:00+00:00           878   
2            v3.2 2025-07-01 00:00:00+00:00           878   
3            v3.2 2025-07-01 00:00:00+00:00    

In [2]:
import pandas as pd

# 1. Flatten the products column into a clean DataFrame
products_list = df['products'].tolist()
df_products_flat = pd.json_normalize(products_list)

print(f"Total flat product records: {len(df_products_flat)}")
print(df_products_flat[['product_id', 'product_name', 'category', 'price.amount', 'stock_qty']].head())

# 2. Check for duplicate product IDs on the FLATTENED dataframe
duplicate_count = df_products_flat['product_id'].duplicated().sum()

print(f"Number of duplicate product IDs: {duplicate_count}")

# 3. View duplicates if any exist
if duplicate_count > 0:
    duplicates = df_products_flat[df_products_flat['product_id'].duplicated(keep=False)]
    print(duplicates[['product_id', 'product_name']])
else:
    print("No duplicate product IDs found! The catalog is clean.")

Total flat product records: 878
  product_id             product_name        category price.amount  stock_qty
0  PRD-00453        Zenith Saree 2024         Apparel    93,078.54        230
1  PRD-00517  Nexon Running Shoes Pro         Apparel    133860.89        407
2  PRD-00524       Meridian Saree Max         Apparel     25610.55        536
3  PRD-00190         Kaveri Microwave  home appliance    163381.89        331
4  PRD-00107  Meridian Smartphone Max     Electronics    157038.82        596
Number of duplicate product IDs: 40
    product_id                 product_name
5    PRD-00506            Tamira Saree Lite
8    PRD-00115      Tamira Air Purifier Pro
22   PRD-00162  Kaveri Air Purifier Classic
34   PRD-00492          Nexon Saree Classic
35   PRD-00410       UrbanCraft Cooking Oil
..         ...                          ...
823  PRD-00376    UrbanCraft Dry Fruits Pro
836  PRD-00824         Kaveri Comic Set Pro
839  PRD-00762          Meridian Novel 2024
857  PRD-00689       Ara

In [3]:
# Drop duplicates based on 'product_id', keeping the first occurrence
df_products_clean = df_products_flat.drop_duplicates(subset=['product_id'], keep='first').copy()

print(f"Original record count: {len(df_products_flat)}")
print(f"Cleaned record count after dropping duplicates: {len(df_products_clean)}")

Original record count: 878
Cleaned record count after dropping duplicates: 838


In [4]:
print(df.isna().sum())

catalog_version    0
generated_at       0
record_count       0
products           0
dtype: int64


In [5]:
import pandas as pd

# 1. Start with your de-duplicated dataframe
df_products_clean = df_products_flat.drop_duplicates(subset=['product_id'], keep='first').copy()

# 2. Clean the price column: remove commas and convert to float
# Check if price.amount is present and is of string type
if 'price.amount' in df_products_clean.columns:
    df_products_clean['price.amount'] = (
        df_products_clean['price.amount']
        .astype(str)
        .str.replace(',', '', regex=True)
        .astype(float)
    )

# 3. Ensure other numerical specs (like weight or stock) are proper numbers
df_products_clean['stock_qty'] = pd.to_numeric(df_products_clean['stock_qty'], errors='coerce')
df_products_clean['specs.weight_kg'] = pd.to_numeric(df_products_clean['specs.weight_kg'], errors='coerce')

print("Data types successfully cleaned!")
print(df_products_clean[['product_id', 'product_name', 'price.amount', 'stock_qty']].head())

# Optional: Save your fully cleaned product catalog for database insertion
df_products_clean.to_csv('cleaned_product_catalog.csv', index=False)

Data types successfully cleaned!
  product_id             product_name  price.amount  stock_qty
0  PRD-00453        Zenith Saree 2024      93078.54        230
1  PRD-00517  Nexon Running Shoes Pro     133860.89        407
2  PRD-00524       Meridian Saree Max      25610.55        536
3  PRD-00190         Kaveri Microwave     163381.89        331
4  PRD-00107  Meridian Smartphone Max     157038.82        596


In [6]:
# Check outliers for price.amount
Q1_price = df_products_clean['price.amount'].quantile(0.25)
Q3_price = df_products_clean['price.amount'].quantile(0.75)
IQR_price = Q3_price - Q1_price
price_outliers = df_products_clean[
    (df_products_clean['price.amount'] < (Q1_price - 1.5 * IQR_price)) | 
    (df_products_clean['price.amount'] > (Q3_price + 1.5 * IQR_price))
]

print(f"Number of price outliers found: {len(price_outliers)}")

# Check outliers for stock_qty
Q1_stock = df_products_clean['stock_qty'].quantile(0.25)
Q3_stock = df_products_clean['stock_qty'].quantile(0.75)
IQR_stock = Q3_stock - Q1_stock
stock_outliers = df_products_clean[
    (df_products_clean['stock_qty'] < (Q1_stock - 1.5 * IQR_stock)) | 
    (df_products_clean['stock_qty'] > (Q3_stock + 1.5 * IQR_stock))
]

print(f"Number of stock quantity outliers found: {len(stock_outliers)}")

Number of price outliers found: 0
Number of stock quantity outliers found: 0


In [7]:
# Force stock quantity to be numeric integers, and replace any negative values with 0 (or drop them)
df_products_flat['stock_qty'] = pd.to_numeric(df_products_flat['stock_qty'], errors='coerce')

# Check for invalid negative or weird float stock quantities
invalid_stock = df_products_flat[df_products_flat['stock_qty'] < 0]
print(f"Found {len(invalid_stock)} records with negative stock.")

# Clean them by dropping or setting to 0
df_products_flat = df_products_flat[df_products_flat['stock_qty'] >= 0].copy()

Found 23 records with negative stock.


In [8]:
# Filter out records where stock quantity is negative or missing
df_products_clean = df_products_flat[df_products_flat['stock_qty'] >= 0].copy()

print(f"Records after removing negative stock anomalies: {len(df_products_clean)}")

Records after removing negative stock anomalies: 855


In [5]:
print(df.columns)

Index(['product_id', 'product_name', 'category', 'sub_category', 'is_fragile',
       'is_hazmat', 'requires_cold_chain', 'stock_qty', 'avg_rating', 'tags',
       'launch_date', 'specs.weight_kg', 'specs.battery_included',
       'price.currency', 'price.amount', 'specs.dimensions_cm.length',
       'specs.dimensions_cm.width', 'specs.dimensions_cm.height', 'specs',
       'image_url'],
      dtype='str')


In [6]:
display(df[['product_name', 'category', 'sub_category']])

,product_name,category,sub_category
0,Zenith Saree 2024,Apparel,Saree
1,Nexon Running Shoes Pro,Apparel,Running Shoes
2,Meridian Saree Max,Apparel,Saree
3,Kaveri Microwave,home appliance,Microwave
4,Meridian Smartphone Max,Electronics,Smartphone
...,...,...,...
833,Tamira Headphones Lite,Electronics,Headphones
834,Zenith Recliner Classic,Furniture,Recliner
835,Meridian Bookshelf Max,Furniture,Bookshelf
836,Tamira Comic Set Pro,Books,Comic Set


In [8]:
# Total unique products, categories, and sub-categories
total_products = df['product_id'].nunique()
total_categories = df['category'].nunique()
total_subcategories = df['sub_category'].nunique()

print(f"📦 Total Unique Products: {total_products}")
print(f"📂 Total Unique Categories: {total_categories}")
print(f"🏷️ Total Unique Sub-Categories: {total_subcategories}\n")

# Display a clean breakdown grouped by Category and Sub-Category with product counts
df_breakdown = df.groupby(['category', 'sub_category']).size().reset_index(name='product_count')
display(df_breakdown)

📦 Total Unique Products: 838
📂 Total Unique Categories: 16
🏷️ Total Unique Sub-Categories: 30



,category,sub_category,product_count
0,Apparel,Jacket,32
1,Apparel,Running Shoes,25
2,Apparel,Saree,30
3,Apparel,T-Shirt,29
4,Books,Comic Set,28
5,Books,Novel,28
6,Books,Textbook,38
7,Electronics,Headphones,24
8,Electronics,Laptop,17
9,Electronics,Smartphone,25


In [9]:
import pandas as pd

# Load your current dataset
df = pd.read_csv('cleaned_product_catalog.csv')

# 1. Sub-Category Image Mapping Function
def assign_photo(row):
    sub = str(row.get('sub_category', '')).lower()
    cat = str(row.get('category', '')).lower()
    
    # Apparel
    if 'saree' in sub:
        return "https://images.unsplash.com/photo-1610030469983-98e550d6193c?w=500"
    elif 'running shoe' in sub or 'shoe' in sub:
        return "https://images.unsplash.com/photo-1542291026-7eec264c27ff?w=500"
    elif 'jacket' in sub:
        return "https://images.unsplash.com/photo-1551028719-00167b16eac5?w=500"
    elif 't-shirt' in sub or 'shirt' in sub:
        return "https://images.unsplash.com/photo-1521572267360-ee0c2909d518?w=500"
        
    # Books
    elif 'novel' in sub or 'comic' in sub or 'book' in sub or 'textbook' in sub:
        return "https://images.unsplash.com/photo-1544716278-ca5e3f4abd8c?w=500"
        
    # Electronics
    elif 'headphone' in sub:
        return "https://images.unsplash.com/photo-1505740420928-5e560c06d30e?w=500"
    elif 'laptop' in sub:
        return "https://images.unsplash.com/photo-1517336714731-489689fd1ca8?w=500"
    elif 'smartphone' in sub or 'phone' in sub:
        return "https://images.unsplash.com/photo-1511707171634-5f897ff02aa9?w=500"
    elif 'smartwatch' in sub or 'watch' in sub:
        return "https://images.unsplash.com/photo-1523275335684-37898b6baf30?w=500"
    elif 'tablet' in sub:
        return "https://images.unsplash.com/photo-1544244015-0df4b3ffc6b0?w=500"
        
    # Furniture
    elif 'bookshelf' in sub or 'chair' in sub or 'table' in sub or 'recliner' in sub:
        return "https://images.unsplash.com/photo-1586023492125-27b2c045efd7?w=500"
        
    # Groceries
    elif 'oil' in sub or 'dry fruit' in sub or 'rice' in sub or 'spice' in sub:
        return "https://images.unsplash.com/photo-1596040033229-a9821ebd058d?w=500"
        
    # Home Appliances
    elif 'air purifier' in sub or 'microwave' in sub or 'mixer' in sub or 'vacuum' in sub:
        return "https://images.unsplash.com/photo-1585659722983-3a675dabf23d?w=500"
        
    # Industrial
    elif 'generator' in sub or 'motor' in sub or 'coil' in sub:
        return "https://images.unsplash.com/photo-1581092160607-ee22621dd758?w=500"
        
    # Pharma
    elif 'insulin' in sub or 'medical' in sub or 'vaccine' in sub:
        return "https://images.unsplash.com/photo-1584308666744-24d5c474f2ae?w=500"
        
    else:
        return "https://images.unsplash.com/photo-1472851294608-062f824d29cc?w=500"

# 2. Realistic Price Normalization Function
def normalize_price(row):
    sub = str(row.get('sub_category', '')).lower()
    
    # Check column name variant for price amount ('price.amount' or 'price_amount')
    price_col = 'price.amount' if 'price.amount' in row else 'price_amount'
    current_price = row.get(price_col, 500)
    
    # Define sensible market price brackets based on sub-category
    if any(k in sub for k in ['oil', 'spice', 'rice bag', 'dry fruits']):
        return round(float(current_price) % 600 + 150, 2)  # ₹150 - ₹750
    elif any(k in sub for k in ['novel', 'comic', 'textbook']):
        return round(float(current_price) % 400 + 200, 2)  # ₹200 - ₹600
    elif any(k in sub for k in ['saree', 'jacket', 't-shirt', 'running shoe']):
        return round(float(current_price) % 1500 + 500, 2)  # ₹500 - ₹2,000
    elif any(k in sub for k in ['headphone', 'smartwatch', 'tablet']):
        return round(float(current_price) % 3000 + 1200, 2)  # ₹1,200 - ₹4,200
    elif any(k in sub for k in ['smartphone', 'laptop', 'generator']):
        return round(float(current_price) % 25000 + 12000, 2) # ₹12,000 - ₹37,000
    elif any(k in sub for k in ['microwave', 'air purifier', 'vacuum cleaner', 'office chair']):
        return round(float(current_price) % 6000 + 2500, 2)  # ₹2,500 - ₹8,500
    else:
        return round(float(current_price) % 2000 + 300, 2)   # General fallback

# Apply updates to dataframe
df['image_url'] = df.apply(assign_photo, axis=1)

price_col_name = 'price.amount' if 'price.amount' in df.columns else 'price_amount'
df[price_col_name] = df.apply(normalize_price, axis=1)

# Save back to CSV
df.to_csv('cleaned_product_catalog.csv', index=False)
print("✅ Successfully updated all 31 sub-category images and normalized market prices!")
display(df[['product_name', 'sub_category', price_col_name, 'image_url']].head(10))

✅ Successfully updated all 31 sub-category images and normalized market prices!


,product_name,sub_category,price.amount,image_url
0,Zenith Saree 2024,Saree,1629.25,https://images.unsplash.com/photo-161003046998...
1,Nexon Running Shoes Pro,Running Shoes,1564.63,https://images.unsplash.com/photo-1542291026-7...
2,Meridian Saree Max,Saree,524.18,https://images.unsplash.com/photo-161003046998...
3,Kaveri Microwave,Microwave,6276.05,https://images.unsplash.com/photo-158565972298...
4,Meridian Smartphone Max,Smartphone,27460.21,https://images.unsplash.com/photo-151170717163...
5,Tamira Saree Lite,Saree,953.84,https://images.unsplash.com/photo-161003046998...
6,UrbanCraft Headphones 2024,Headphones,3066.34,https://images.unsplash.com/photo-150574042092...
7,Meridian Novel Classic,Novel,580.33,https://images.unsplash.com/photo-1544716278-c...
8,Tamira Air Purifier Pro,Air Purifier,4975.77,https://images.unsplash.com/photo-158565972298...
9,Tamira Saree,Saree,1640.53,https://images.unsplash.com/photo-161003046998...
